# 🧠 DSA Lab 06: Recursion, Backtracking and Divide & Conquer
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 06 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 📖 Part A: Anatomy of a Recursive Function

Every correct recursive function has **two parts**:

| Part | Role | Factorial example |
|---|---|---|
| Base case | The smallest input, answered directly; it **stops** the recursion | 0! = 1 |
| Recursive case | Reduces the problem and calls itself on a **smaller** input | n! = n × (n − 1)! |

In [ ]:
def factorial(n):
    if n == 0:                     # base case
        return 1
    return n * factorial(n - 1)    # recursive case (smaller input)

print(factorial(5))

> **⚠️ No base case = infinite recursion**
>
> If the base case is missing or never reached, the function calls itself forever until Python stops it with `RecursionError: maximum recursion depth exceeded`.

### The Call Stack

Each function call gets a **stack frame** that stores its own local variables. Calls are **pushed** onto the call stack and **popped** when they return (Last In, First Out, like the Stack you will build in Lab 10).

**Winding (going down) and unwinding (coming back)**

```text
factorial(4)
  -> 4 * factorial(3)
         -> 3 * factorial(2)
                -> 2 * factorial(1)
                       -> 1 * factorial(0)
                              -> 1            (base case)
                       <- 1 * 1 = 1
                <- 2 * 1 = 2
         <- 3 * 2 = 6
  <- 4 * 6 = 24
```

In [ ]:
def factorial_trace(n, depth=0):
    indent = "    " * depth
    print(f"{indent}factorial({n}) called")
    if n == 0:
        print(f"{indent}base case -> 1")
        return 1
    result = n * factorial_trace(n - 1, depth + 1)
    print(f"{indent}factorial({n}) returns {result}")
    return result

factorial_trace(4)

### How to Think Recursively (The Leap of Faith)

1. **Define** exactly what the function returns, e.g., “`sum_list(a)` returns the sum of list a”.
2. **Base case:** what is the answer for the smallest input (empty list → 0)?
3. **Trust** that the function already works for a smaller input (`sum_list(a[1:])`).
4. **Combine:** use that smaller answer to build the full answer (`a[0] + sum_list(a[1:])`).

In [ ]:
def sum_list(a):
    if not a:                      # base case: empty list
        return 0
    return a[0] + sum_list(a[1:])  # first element + sum of the rest

def count_down(n):
    if n == 0:
        print("Go! 🚀")
        return
    print(n)
    count_down(n - 1)

print(sum_list([4, 8, 15, 16, 23, 42]))
count_down(3)

## 🧮 Part B: Classic Recursive Problems

### Power: xⁿ

In [ ]:
def power(x, n):                   # O(n)
    if n == 0:
        return 1
    return x * power(x, n - 1)

def fast_power(x, n):              # O(log n): halve the exponent
    if n == 0:
        return 1
    half = fast_power(x, n // 2)
    return half * half if n % 2 == 0 else half * half * x

print(power(2, 10), fast_power(2, 10), fast_power(3, 13) == 3 ** 13)

### Recursive Binary Search

In [ ]:
def binary_search_rec(arr, target, low, high):
    if low > high:                 # base case: empty range
        return -1
    mid = (low + high) // 2
    if arr[mid] == target:
        return mid
    if arr[mid] < target:
        return binary_search_rec(arr, target, mid + 1, high)
    return binary_search_rec(arr, target, low, mid - 1)

data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print(binary_search_rec(data, 72, 0, len(data) - 1))

### Tower of Hanoi

Move n discs from peg A to peg C using peg B, moving one disc at a time and never placing a bigger disc on a smaller one. The recursive insight: **move n−1 discs out of the way, move the biggest disc, then move the n−1 discs on top of it**.

In [ ]:
def hanoi(n, source, helper, target):
    if n == 0:
        return 0
    moves = hanoi(n - 1, source, target, helper)   # move n-1 to helper
    print(f"Move disc {n} from {source} to {target}")
    moves += 1
    moves += hanoi(n - 1, helper, source, target)  # move n-1 onto target
    return moves

print("Total moves:", hanoi(3, "A", "B", "C"))

The number of moves is **2ⁿ − 1**. For 64 discs (the legend), that is about 1.8 × 10¹⁹ moves: at one move per second it would take around 585 billion years. A perfect example of **exponential** growth.

## 🌳 Part C: Recursion Trees and Memoization

### Fibonacci: The Danger of Repeated Work

Fibonacci numbers: F(0) = 0, F(1) = 1, F(n) = F(n−1) + F(n−2). The direct recursive code is beautiful but **extremely slow**.

**Recursion tree for fib(5)**

```text
fib(5)
├── fib(4)
│   ├── fib(3)
│   │   ├── fib(2) ── fib(1), fib(0)
│   │   └── fib(1)
│   └── fib(2) ── fib(1), fib(0)        <- fib(2) computed again
└── fib(3)                               <- whole fib(3) computed again!
    ├── fib(2) ── fib(1), fib(0)
    └── fib(1)
```

In [ ]:
calls = 0
def fib_naive(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

for n in [10, 20, 25]:
    calls = 0
    value = fib_naive(n)
    print(f"fib({n}) = {value:>6}  ->  {calls:>7,} function calls")

The number of calls roughly **doubles** with each increase of n: the complexity is **O(2ⁿ)**. `fib_naive(50)` would need about 40 billion calls.

### Memoization: Remember What You Already Computed

**Memoization** stores the result of each call in a dictionary. Before computing, check whether the answer is already known. Each F(k) is computed **once**, so the complexity drops to **O(n)**.

In [ ]:
def fib_memo(n, memo=None):
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]             # already computed: O(1)
    if n < 2:
        return n
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

from functools import lru_cache

@lru_cache(maxsize=None)           # Python's built-in memoization
def fib_cached(n):
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)

print(fib_memo(80), fib_cached(80))

### Recursion vs Iteration

In [ ]:
def fib_iter(n):                   # O(n) time, O(1) space, no recursion limit
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

import sys
print(fib_iter(80), "| Python recursion limit:", sys.getrecursionlimit())

|  | Recursion | Iteration (loops) |
|---|---|---|
| Code | Short, close to the mathematical definition | Sometimes longer |
| Memory | O(depth) for the call stack | Usually O(1) |
| Risk | Stack overflow / RecursionError (~1000 deep in Python) | Infinite loop |
| Best for | Trees, graphs, divide & conquer, backtracking | Simple repetition, very deep processes |

## 🔙 Part D: Backtracking

**Backtracking** builds a solution step by step, explores one choice, and **undoes** (backtracks) that choice to try the next one. It systematically explores all possibilities.

#### Generate all subsets (e.g., all possible combinations of toppings)

In [ ]:
def subsets(items):
    result = []
    current = []

    def explore(i):
        if i == len(items):                  # decided for every item
            result.append(current[:])
            return
        current.append(items[i])             # choice 1: include items[i]
        explore(i + 1)
        current.pop()                        # undo (backtrack)
        explore(i + 1)                       # choice 2: exclude items[i]

    explore(0)
    return result

print(subsets(["cheese", "olives", "chicken"]))

Each item has 2 choices, so n items give **2ⁿ** subsets. Backtracking is exponential, but it is often the only exact method, and it can be sped up by **pruning** branches that cannot lead to a solution (e.g., in Sudoku).

## ✂️ Part E: Divide & Conquer

**Divide & conquer** solves a problem in three steps:

1. **Divide** the problem into smaller sub-problems (usually halves).
2. **Conquer** each sub-problem recursively.
3. **Combine** the sub-answers into the final answer.

In [ ]:
def max_dc(a, low, high):
    if low == high:                          # one element
        return a[low]
    mid = (low + high) // 2
    left_max = max_dc(a, low, mid)           # conquer left half
    right_max = max_dc(a, mid + 1, high)     # conquer right half
    return left_max if left_max > right_max else right_max   # combine

marks = [67, 89, 45, 92, 78, 55, 81]
print(max_dc(marks, 0, len(marks) - 1))

| Algorithm | Divide | Combine | Complexity |
|---|---|---|---|
| Binary search | Pick one half | Nothing | O(log n) |
| Max by D&C | Two halves | Compare two values | O(n) |
| Merge sort (Lab 07) | Two halves | Merge two sorted halves: O(n) | O(n log n) |
| Quick sort (Lab 07) | Partition around a pivot | Nothing | O(n log n) average |

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <unordered_map>
using namespace std;

long long factorial(int n) {
    if (n == 0) return 1;            // base case
    return n * factorial(n - 1);     // recursive case
}

unordered_map<int, long long> memo;
long long fib(int n) {
    if (n < 2) return n;
    if (memo.count(n)) return memo[n];
    return memo[n] = fib(n - 1) + fib(n - 2);
}

void hanoi(int n, char from, char via, char to) {
    if (n == 0) return;
    hanoi(n - 1, from, to, via);
    cout << "Move disc " << n << " from " << from << " to " << to << endl;
    hanoi(n - 1, via, from, to);
}

int main() {
    cout << factorial(10) << " " << fib(80) << endl;
    hanoi(3, 'A', 'B', 'C');
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Sum of Digits
Write a **recursive** function `digit_sum(n)` that returns the sum of the digits of a non-negative integer. No loops, no `str()`.

```text
digit_sum(2026) → 10
```

In [ ]:
def digit_sum(n):
    # TODO: base case (single digit) and recursive case (last digit + rest)
    pass

In [ ]:
# Test cell: run after your solution
assert digit_sum(2026) == 10
assert digit_sum(0) == 0
assert digit_sum(9) == 9
assert digit_sum(999999) == 54
print("✅ Task 1 passed")

### Task 2 🟢 Reverse a String Recursively
Write `reverse(s)` recursively, without slicing tricks like `[::-1]` and without loops. Hint: reverse of s = reverse of `s[1:]` + `s[0]`.

```text
reverse("Okara") → "arakO"
```

In [ ]:
def reverse(s):
    # TODO
    pass

In [ ]:
# Test cell: run after your solution
assert reverse("Okara") == "arakO"
assert reverse("") == ""
assert reverse("a") == "a"
assert reverse("racecar") == "racecar"
print("✅ Task 2 passed")

### Task 3 🟡 Count Files in a Folder Tree
A folder is represented as a dictionary: keys are names; a value of `None` means a file, and a dictionary means a sub-folder. Write `count_files(folder)` and `total_depth(folder)` (the maximum nesting depth; a folder with only files has depth 1).

This is exactly how your operating system walks directories.

```text
{"notes.txt": None, "labs": {"lab1.py": None, "data": {"a.csv": None}}}
count_files → 3,  total_depth → 3
```

In [ ]:
def count_files(folder):
    # TODO: files count 1; sub-folders are counted recursively
    pass

def total_depth(folder):
    # TODO: 1 + the deepest sub-folder depth
    pass

In [ ]:
# Test cell: run after your solution
tree = {"notes.txt": None, "labs": {"lab1.py": None, "data": {"a.csv": None}}}
assert count_files(tree) == 3 and total_depth(tree) == 3
assert count_files({}) == 0 and total_depth({}) == 1
assert count_files({"x": {}, "y": {"z": {}}}) == 0 and total_depth({"x": {}, "y": {"z": {}}}) == 3
print("✅ Task 3 passed")

### Task 4 🟡 Grid Paths with Memoization
A delivery rider starts at the top-left of an r × c grid of streets and can move only **right** or **down**. Some cells are blocked (`1`). Count the number of paths to the bottom-right.

Write the recursive solution `paths(grid)`, then add memoization so that a 15 × 15 grid finishes instantly.

```text
grid = [[0,0,0],
        [0,1,0],
        [0,0,0]]   → 2
```

In [ ]:
def paths(grid):
    r, c = len(grid), len(grid[0])
    memo = {}

    def count(i, j):
        # TODO: out of grid or blocked -> 0; destination -> 1
        # TODO: memo check, then count(i+1, j) + count(i, j+1)
        pass

    return count(0, 0)

In [ ]:
# Test cell: run after your solution
assert paths([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) == 2
assert paths([[0, 1], [0, 0]]) == 1
assert paths([[1]]) == 0
assert paths([[0] * 15 for _ in range(15)]) == 40116600
print("✅ Task 4 passed")

### Task 5 🟡 Tower of Hanoi Move List
Write `hanoi_moves(n, a, b, c)` that **returns** the list of moves as tuples `(disc, from, to)` instead of printing. Verify that the length is 2ⁿ − 1 and that no move ever places a bigger disc on a smaller one (write a simulator).

```text
hanoi_moves(2, "A", "B", "C") → [(1, "A", "B"), (2, "A", "C"), (1, "B", "C")]
```

In [ ]:
def hanoi_moves(n, a, b, c):
    # TODO: moves(n-1 from a to b) + [(n, a, c)] + moves(n-1 from b to c)
    pass

In [ ]:
# Test cell: run after your solution
assert hanoi_moves(2, "A", "B", "C") == [(1, "A", "B"), (2, "A", "C"), (1, "B", "C")]
for n in range(1, 9):
    moves = hanoi_moves(n, "A", "B", "C")
    assert len(moves) == 2 ** n - 1
    pegs = {"A": list(range(n, 0, -1)), "B": [], "C": []}
    for disc, f, t in moves:
        assert pegs[f][-1] == disc
        assert not pegs[t] or pegs[t][-1] > disc
        pegs[t].append(pegs[f].pop())
    assert pegs["C"] == list(range(n, 0, -1))
print("✅ Task 5 passed")

### Task 6 🔴 Permutations (Backtracking)
Write `permutations(items)` that returns all orderings of a list using backtracking with a `used` list. Do not use `itertools`. The result order should be lexicographic if the input is sorted.

```text
permutations([1, 2, 3]) → [[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]
```

In [ ]:
def permutations(items):
    result, current = [], []
    used = [False] * len(items)

    def explore():
        # TODO: if current is complete, save a copy
        # TODO: try every unused item, recurse, then undo
        pass

    explore()
    return result

In [ ]:
# Test cell: run after your solution
import itertools
assert permutations([1, 2, 3]) == [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
assert permutations([]) == [[]]
assert len(permutations(list(range(6)))) == 720
assert permutations(list("abcd")) == [list(p) for p in itertools.permutations("abcd")]
print("✅ Task 6 passed")

### Task 7 🔴 Flood Fill: Count Lakes on a Map
A satellite map is a grid where `1` is water and `0` is land. Water cells connected **up, down, left or right** form one lake. Write `count_lakes(grid)` using a recursive `fill(i, j)` that marks a whole lake as visited.

This is the same algorithm used by the paint-bucket tool in MS Paint and by region detection in remote-sensing images.

```text
[[1,1,0,0],
 [1,0,0,1],
 [0,0,1,1],
 [0,0,0,0]]   → 2
```

In [ ]:
def count_lakes(grid):
    g = [row[:] for row in grid]
    rows = len(g)
    cols = len(g[0]) if rows else 0

    def fill(i, j):
        # TODO: stop if outside the grid or not water; else mark and recurse 4 ways
        pass

    lakes = 0
    # TODO: scan every cell; each unvisited water cell starts a new lake
    return lakes

In [ ]:
# Test cell: run after your solution
m = [[1, 1, 0, 0], [1, 0, 0, 1], [0, 0, 1, 1], [0, 0, 0, 0]]
assert count_lakes(m) == 2 and m[0][0] == 1
assert count_lakes([[0, 0], [0, 0]]) == 0
assert count_lakes([[1, 0, 1], [0, 1, 0], [1, 0, 1]]) == 5
assert count_lakes([]) == 0
print("✅ Task 7 passed")

## 🔬 Bonus: Draw a Recursive Fractal Tree

In [ ]:
# Bonus: run after completing the tasks
try:
    import math
    import matplotlib.pyplot as plt

    def branch(x, y, angle, length, depth, ax):
        if depth == 0:
            return
        x2 = x + length * math.cos(math.radians(angle))
        y2 = y + length * math.sin(math.radians(angle))
        ax.plot([x, x2], [y, y2], color="green" if depth < 3 else "saddlebrown", lw=depth * 0.6)
        branch(x2, y2, angle - 25, length * 0.72, depth - 1, ax)   # left branch
        branch(x2, y2, angle + 25, length * 0.72, depth - 1, ax)   # right branch

    fig, ax = plt.subplots(figsize=(6, 6))
    branch(0, 0, 90, 10, 9, ax)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title("A tree drawn by recursion (depth 9)")
    plt.show()
except ImportError:
    print("matplotlib is not installed here; it is available in Google Colab.")

---
## 🎉 Lab 06 Complete!

Next: **Lab 07: Efficient Sorting (Merge Sort, Quick Sort, Counting Sort)**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab06/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)